# Palace Worflow Test
This notebook contains a simple `pcLab` -> `gds2palace` workflow test.

In [1]:
import pclab as pl
import gdspy, gds2palace as gp, gdsfactory as gf
import math, os, subprocess, pathlib, sys
import ihp

ihp.PDK.activate()

tech = pl.Technology("../lib/pcLab/examples_SG13G2/SG13G2.tech")
ind  = pl.inductorSym(tech)

gdspy.current_library = gdspy.GdsLibrary()
gf.clear_cache()

In [2]:
## Add the scripts folder to path
notebook_root = pathlib.Path(os.getcwd())
repo_root     = notebook_root.parent.absolute()
scripts_root  = repo_root / pathlib.Path("scripts") 

print(str(scripts_root))
os.environ["PATH"] += os.pathsep + os.pathsep.join([str(scripts_root)])
print(os.environ["PATH"])

os.environ["DISPLAY"] = "localhost:10.0"

/home/simon/GitRepos/InductorLab/scripts
/home/simon/.pyenv/versions/InductorLab/bin:/home/simon/.pyenv/versions/3.12.12/envs/InductorLab/bin:/home/simon/.vscode-server/cli/servers/Stable-ce099c1ed25d9eb3076c11e4a280f3eb52b4fbeb/server/bin/remote-cli:/home/simon/.local/bin:/home/simon/.local/bin:/home/simon/.local/bin:/home/simon/.pyenv/plugins/pyenv-virtualenv/shims:/home/simon/.pyenv/shims:/home/simon/.pyenv/bin:/home/simon/.local/bin:/home/simon/.local/bin:/home/simon/.nix-profile/bin:/nix/var/nix/profiles/default/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/home/simon/GitRepos/InductorLab/scripts


In [3]:
signal_layer = "Metal5"
underpass_layer = "Metal4"

L_target = 1e-9  # 1 nano henry
w = 5            # microns
s = 2            # microns
N = 2
geom = "octagon"

# Calculate the minimum diameter for this inductor (Wheeler's equation)
d_outer = pl.calculate_octa_diameter (N, w, s, L_target) 
d_outer = math.ceil(d_outer*100)/100

print(f"Target inductance {L_target*1e9} nH at N={N} w={w} s={s} => outer diameter {d_outer} microns")

Target inductance 1.0 nH at N=2 w=5 s=2 => outer diameter 131.85 microns


In [ ]:
# Check if geometry is valid and generate the GDSII file
valid = ind.setupGeometry(d_outer, w, s, N, signal_layer, underpass_layer, geom)
ind_name = "inductorSym_" + geom + "_d_outer" + str(d_outer) + "_w" +  str(w) + "_n" + str(N) + "_s" + str(s)
out_file = ".out/" + ind_name + '.gds'

if valid:
    ind.genGeometry()
    ind.genGDSII(out_file, structName = ind_name)
    print(f"Created output file {ind_name}.gds")
    # Create a gds2palace compatible file also
    pl.gds_pin2viaport(out_file, width=w)
    em_gds = ".out/" + ind_name + '_forEM' + '.gds'
else:
    d_outer_min = ind.get_min_diameter()
    print(f"Could not create layout, required d_outer {d_outer} is smaller than minimum possible value {d_outer_min}")


Created output file inductorSym_octagon_d_outer131.85_w5_n2_s2.gds


In [5]:
component = gf.Component()
component = gf.read.import_gds(em_gds)

component.show()

2026-03-10 16:18:08.298 | WARNING  | kfactory.kcell:show:3980 - Could not connect to klive server


## `gds2palace` flow

In [6]:
# ======================== workflow settings ================================

#start solver after creating the model?
start_simulation = True
run_command = ['./run_sim']   

In [7]:
# ===================== input files and path settings =======================

gds_filename = em_gds
XML_filename = "../resources/SG13G2_200um.xml"

# preprocess GDSII for safe handling of cutouts/holes?
preprocess_gds = True

# merge via polygons with distance less than .. microns, set to 0 to disable via merging.
merge_polygon_size = 2

# set and create directory for simulation output
sim_path = gp.utilities.create_sim_path(".out/", ind_name)
print('Simulation data directory: ', sim_path)

Simulation data directory:  .out/palace_model/inductorSym_octagon_d_outer131.85_w5_n2_s2_data


In [8]:
# ======================== simulation settings ================================


settings = {}

settings['unit']   = 1e-6  # geometry is in microns
settings['margin'] = 150    # distance in microns from GDSII geometry boundary to simulation boundary 
settings['air_around'] = 50

settings['fstart']  = 0e9
settings['fstop']   = 50e9
settings['fstep']   = 0.5e9

settings["fdump"] = [15e9]  # save field dump at these frequency points 
# settings["fpoint"] = [15e9] # discrete frequency points, no field dump

settings['refined_cellsize'] = 5  # mesh cell size in conductor region
settings['cells_per_wavelength'] = 10   # how many mesh cells per wavelength, must be 10 or more

settings['meshsize_max'] = 70  # microns, override cells_per_wavelength 
settings['adaptive_mesh_iterations'] = 0

settings['no_gui'] = False


# Ports from GDSII Data, polygon geometry from specified special layer
# Excitations can be switched off by voltage=0, those S-parameter will be incomplete then

simulation_ports = gp.simulation_setup.all_simulation_ports()
# instead of in-plane port specified with target_layername, we here use via port specified with from_layername and to_layername
simulation_ports.add_port(gp.simulation_setup.simulation_port(portnumber=1, voltage=1, port_Z0=50, source_layernum=201, from_layername='Metal5', to_layername='TopMetal2', direction='z'))
simulation_ports.add_port(gp.simulation_setup.simulation_port(portnumber=2, voltage=1, port_Z0=50, source_layernum=202, from_layername='Metal5', to_layername='TopMetal2', direction='z'))


In [9]:
# ======================== simulation ================================

# get technology stackup data
materials_list, dielectrics_list, metals_list = gp.stackup_reader.read_substrate(XML_filename)
# get list of layers from technology
layernumbers = metals_list.getlayernumbers()
layernumbers.extend(simulation_ports.portlayers)

# read geometries from GDSII, only purpose 0
gdspy.current_library = gdspy.GdsLibrary()
gf.clear_cache()
allpolygons = gp.gds_reader.read_gds(gds_filename, layernumbers, purposelist=[0], metals_list=metals_list, preprocess=preprocess_gds, merge_polygon_size=merge_polygon_size)


########### create model ###########

settings['simulation_ports'] = simulation_ports
settings['materials_list'] = materials_list
settings['dielectrics_list'] = dielectrics_list
settings['metals_list'] = metals_list
settings['layernumbers'] = layernumbers
settings['allpolygons'] = allpolygons
settings['sim_path'] = sim_path 
settings['model_basename'] = ind_name 


# list of ports that are excited (set voltage to zero in port excitation to skip an excitation!)
excite_ports = simulation_ports.all_active_excitations()
config_name, data_dir = gp.simulation_setup.create_palace (excite_ports, settings)


# for convenience, write run script to model directory
gp.utilities.create_run_script(sim_path)


if start_simulation:
    try:
        os.chdir(sim_path)
        subprocess.run(run_command, shell=True)
    except:
        print(f"Unable to run Palace using command ",run_command)
    finally:
        # Go back the the original working directory
        os.chdir(notebook_root)

Reading GDSII input file: .out/inductorSym_octagon_d_outer131.85_w5_n2_s2_forEM.gds
Pre-processing GDSII to handle cutouts and self-intersecting polygons
Using boundary condition  ['ABC', 'ABC', 'ABC', 'ABC', 'ABC', 'ABC']
Starting to create mesh file and config file
---------------------------------------------------
Wavelength in air: 6000.0 units
  meshsize_max: 70.0  units
  max_cellsize_air: 600.0 units
---------------------------------------------------
Adding metal tags ...
Adding ports ...                                                                                                                          
Adding dielectrics ...
Dielectric =  AIRFragments - Adding holes                                                                                
Dielectric =  Passive
Dielectric =  SiO2
Dielectric =  EPI
Dielectric =  Substrate
Dielectric =  airbox
Dielectric  AIR  with max_cellsize_local =  70 units
Dielectric  Passive  with max_cellsize_local =  70 units
Dielectric  SiO2

Info    : Done writing '.out/palace_model/inductorSym_octagon_d_outer131.85_w5_n2_s2_data/inductorSym_octagon_d_outer131.85_w5_n2_s2.msh'
-------------------------------------------------------
Version       : 4.15.1
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 20260216
Build host    : gmsh.info
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blas[petsc] Blossom Cgns DIntegration Dlopen DomHex Eigen[contrib] Fltk Gmm[contrib] Hxt Jpeg Kbipack Lapack[petsc] LinuxJoystick MathEx[contrib] Med Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom PETSc Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.11
PETSc version : 3.14.4 (real arithmtic)
OCC version   : 7.8.1
MED version   : 4.1.0
Packaged by   : geuzaine
Web site      : https://gmsh.info
Issue tracker : https

/home/simon/GitRepos/InductorLab/lib/gds2palace/scripts/combine_extend_snp.py:282: FutureWarning: skrf.connect is deprecated. Please import connect from skrf.network instead.
  ntwk = rf.connect(inductor, 0, ntwk, 0)
